# This notebook serves as a template for cleaning the original text data and saving the train/test splits to Parquet files rather than csv

In [1]:
# !pip install nltk

# !pip install pyspellchecker

# !pip install textblob

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [2]:
train.head()

,essay_id,full_text,score
0,000d118,Many people have car where they live. The thin...,3
1,000fe60,I am a scientist at NASA that is discussing th...,3
2,001ab80,People always wish they had the same technolog...,4
3,001bdc0,"We all heard about Venus, the planet without a...",4
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3


In [3]:
test.head()

,essay_id,full_text
0,000d118,Many people have car where they live. The thin...
1,000fe60,I am a scientist at NASA that is discussing th...
2,001ab80,People always wish they had the same technolog...


In [4]:
# def preprocess_text(df, col = 'full_text'):

#     df['paragraph'] = df[col].str.split('\n\n')

#     return df


# train = preprocess_text(train)

# train.head()

In [5]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
from spellchecker import SpellChecker
from textblob import TextBlob

# # Set the download location to Kaggle's working directory
# download_dir = '/kaggle/working/nltk_data'

# # Add this download directory to nltk's data path
# if download_dir not in nltk.data.path:
#     nltk.data.path.append(download_dir)

# Download necessary datasets from NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')
nltk.download('universal_tagset')
nltk.download('maxent_ne_chunker')
nltk.download('words')

# Now check if the directory is correctly set and files are present
# import os
# print(os.listdir(download_dir))  # This should show the downloaded files


[nltk_data] Downloading package punkt to /home/laptop/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /home/laptop/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/laptop/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/laptop/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package omw-1.4 to /home/laptop/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/laptop/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /home/laptop/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nl

True

In [6]:
# import zipfile
# import os

# # Path to the zip file
# zip_path = '/kaggle/working/nltk_data/corpora/wordnet.zip'

# # Target extraction directory
# extract_dir = '/kaggle/working/nltk_data/corpora/wordnet'

# # Create the target directory if it doesn't already exist
# if not os.path.exists(extract_dir):
#     os.makedirs(extract_dir)

# # Extract the zip file
# with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

# # Verify the files have been extracted
# print(os.listdir(extract_dir))  # Should show the contents of the wordnet corpus


In [7]:
# import os
# import shutil

# # Set the source and target directories
# source_dir = '/kaggle/working/nltk_data/corpora/wordnet/wordnet'
# target_dir = '/kaggle/working/nltk_data/corpora/wordnet'

# # Move each file and subdirectory from the source to the target directory
# for filename in os.listdir(source_dir):
#     source_file = os.path.join(source_dir, filename)
#     target_file = os.path.join(target_dir, filename)
#     if os.path.isdir(source_file):
#         if os.path.exists(target_file):
#             shutil.rmtree(target_file)  # Remove if target directory already exists
#         shutil.move(source_file, target_dir)
#     else:
#         if os.path.exists(target_file):
#             os.remove(target_file)  # Remove if target file already exists
#         shutil.move(source_file, target_dir)

# # Clean up the now empty source directory
# os.rmdir(source_dir)

# # Verify the structure
# print(os.listdir(target_dir))  # Should list 'lexnames' among other files


In [8]:
# import os
# from tqdm import tqdm

# # List of file paths to unzip
# embeddings = ['data/archive.zip']

# def unzip_embeddings(file_paths):
#     """
#     Unzip a list of files.

#     Args:
#         file_paths (list): List of file paths to unzip.

#     Returns:
#         None
#     """
#     # Initialize tqdm with the total number of files to unzip
#     with tqdm(total=len(file_paths)) as pbar:
#         for emb in file_paths:
#             # Use the -o flag to automatically replace files
#             if os.system(f'unzip -o {emb} -d data/') == 0:
#                 print(f"Inflating {emb} successful.")
#             else:
#                 print(f"Inflating {emb} failed.")
#             pbar.update(1)  # Update the progress bar

# # Call the function to unzip files
# unzip_embeddings(embeddings)

In [9]:
def embedding_checks(df, glove_path, paragram_path, wiki_news_path, col_name = 'clean_text'):

    """
    Preprocesses text for both training and testing datasets. 
    Includes loading embeddings, building vocabularies, cleaning text among other things.
    
    :param summaries_train: DataFrame with the training data
    :param summaries_test: DataFrame with the testing data
    :param glove_path: path to the GloVe embedding
    :param paragram_path: path to the Paragram embedding
    :param wiki_news_path: path to the Wiki News embedding
    
    :return: Preprocessed DataFrame and list of out-of-vocab words
    """

    import tqdm 

    def load_embed(file):
        """
        Load the embeddings from a file.
        """
        print(f"Loading embeddings from {file}")
        
        def get_coefs(word, *arr): 
            return word, np.asarray(arr, dtype='float32')
        
        if file == wiki_news_path:
            embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file), "Reading Embedding File") if len(o)>100)
        else:
            embeddings_index = dict(get_coefs(*o.split(" ")) for o in tqdm(open(file, encoding='latin'), "Reading Embedding File"))
        
        print(f"Loaded embeddings from {file}")
        return embeddings_index

    # Load embeddings
    print("Loading all embeddings.")
    embed_glove = load_embed(glove_path)
    embed_paragram = load_embed(paragram_path)
    embed_fasttext = load_embed(wiki_news_path)
    print("All embeddings loaded.")
    
    def build_vocab(texts):
        """
        Build a vocabulary from a given list of texts.
        """
        print("Building vocabulary.")
        sentences = texts.apply(lambda x: x.split()).values
        vocab = {}
        for sentence in tqdm(sentences, "Populating Vocabulary"):
            for word in sentence:
                try:
                    vocab[word] += 1
                except KeyError:
                    vocab[word] = 1
        print("Vocabulary built.")
        return vocab

    def check_coverage(vocab, embeddings_index):
        """
        Check which words in the vocabulary are covered by the embeddings.
        """
        print("Checking coverage.")
        known_words = {}
        unknown_words = {}
        nb_known_words = 0
        nb_unknown_words = 0
        for word in tqdm(vocab.keys(), "Checking Words"):
            try:
                known_words[word] = embeddings_index[word]
                nb_known_words += vocab[word]
            except:
                unknown_words[word] = vocab[word]
                nb_unknown_words += vocab[word]
                pass
        unknown_words = sorted(unknown_words.items(), key=operator.itemgetter(1))[::-1]
        print("Coverage checked.")
        return unknown_words

    # Build and check vocab for train and test datasets
    print("Processing train and test datasets.")
    
    vocab_train = build_vocab(df[col_name])
    
    oov_glove_train = check_coverage(vocab_train, embed_glove)
    oov_paragram_train = check_coverage(vocab_train, embed_paragram)
    oov_fasttext_train = check_coverage(vocab_train, embed_fasttext)
  
    print("Processed train and test datasets.")
    
    
    # Rebuild and check vocab after cleaning contractions
    
#     vocab_train_clean = build_vocab(df['clean_text'])
    
#     oov_glove_train = check_coverage(vocab_train_clean, embed_glove)
#     oov_paragram_train = check_coverage(vocab_train_clean, embed_paragram)
#     oov_fasttext_train = check_coverage(vocab_train_clean, embed_fasttext)


    return df, oov_glove_train, oov_paragram_train, oov_fasttext_train

In [10]:
# Preprocessing
import numpy as np
import pandas as pd
import operator
from spellchecker import SpellChecker
from tqdm import tqdm  # Import tqdm
import re

def clean_text(df, col_name = 'full_text'):
    """
    Preprocesses text for both training and testing datasets. 
    Includes loading embeddings, building vocabularies, cleaning text among other things.
    
    :param summaries_train: DataFrame with the training data
    :param summaries_test: DataFrame with the testing data
    :param glove_path: path to the GloVe embedding
    :param paragram_path: path to the Paragram embedding
    :param wiki_news_path: path to the Wiki News embedding
    
    :return: Preprocessed DataFrame and list of out-of-vocab words
    """
    print("Starting text cleaning process. \n")


    
    # Lowercase all texts

    df['lowered'] = df[col_name].apply(lambda x: x.lower())

    
    # Handle contractions
    contraction_mapping = {"ain't": "is not", "aren't": "are not","can't": "cannot", "'cause": "because", "could've": "could have", "couldn't": "could not", 
                           "didn't": "did not",  "doesn't": "does not", "don't": "do not", "hadn't": "had not", "hasn't": "has not", "haven't": "have not", 
                           "he'd": "he would","he'll": "he will", "he's": "he is", "how'd": "how did", "how'd'y": "how do you", "how'll": "how will", "how's": "how is",  
                           "I'd": "I would", "I'd've": "I would have", "I'll": "I will", "I'll've": "I will have","I'm": "I am", "I've": "I have", "i'd": "i would", 
                           "i'd've": "i would have", "i'll": "i will",  "i'll've": "i will have","i'm": "i am", "i've": "i have", "isn't": "is not", "it'd": "it would", 
                           "it'd've": "it would have", "it'll": "it will", "it'll've": "it will have","it's": "it is", "let's": "let us", "ma'am": "madam", 
                           "mayn't": "may not", "might've": "might have","mightn't": "might not","mightn't've": "might not have", "must've": "must have", 
                           "mustn't": "must not", "mustn't've": "must not have", "needn't": "need not", "needn't've": "need not have","o'clock": "of the clock", 
                           "oughtn't": "ought not", "oughtn't've": "ought not have", "shan't": "shall not", "sha'n't": "shall not", "shan't've": "shall not have", 
                           "she'd": "she would", "she'd've": "she would have", "she'll": "she will", "she'll've": "she will have", "she's": "she is", 
                           "should've": "should have", "shouldn't": "should not", "shouldn't've": "should not have", "so've": "so have","so's": "so as", 
                           "this's": "this is","that'd": "that would", "that'd've": "that would have", "that's": "that is", "there'd": "there would", 
                           "there'd've": "there would have", "there's": "there is", "here's": "here is","they'd": "they would", "they'd've": "they would have", 
                           "they'll": "they will", "they'll've": "they will have", "they're": "they are", "they've": "they have", "to've": "to have", 
                           "wasn't": "was not", "we'd": "we would", "we'd've": "we would have", "we'll": "we will", "we'll've": "we will have", "we're": "we are", 
                           "we've": "we have", "weren't": "were not", "what'll": "what will", "what'll've": "what will have", "what're": "what are",  "what's": "what is",
                           "what've": "what have", "when's": "when is", "when've": "when have", "where'd": "where did", "where's": "where is", "where've": "where have",
                           "who'll": "who will", "who'll've": "who will have", "who's": "who is", "who've": "who have", "why's": "why is", "why've": "why have", 
                           "will've": "will have", "won't": "will not", "won't've": "will not have", "would've": "would have", "wouldn't": "would not", 
                           "wouldn't've": "would not have", "y'all": "you all", "y'all'd": "you all would","y'all'd've": "you all would have","y'all're": "you all are",
                           "y'all've": "you all have","you'd": "you would", "you'd've": "you would have", "you'll": "you will", "you'll've": "you will have", 
                           "you're": "you are", "you've": "you have" }
    
    
    def clean_contractions(text, mapping):
        """
        Replace contractions in the text based on a given mapping.
        
        :param text: The original text
        :param mapping: Dictionary containing contractions mapping
        
        :return: Text with contractions replaced
        """
        
        specials = ["’", "‘", "´", "`"]
        
        for s in specials:
            text = text.replace(s, "'")
        text = ' '.join([mapping[t] if t in mapping else t for t in text.split(" ")])
        return text

    # Apply contraction cleaning to train and test datasets

    df['cleaned_text'] = df['lowered'].apply(lambda x: clean_contractions(x, contraction_mapping))
    
    
        
    punct = "/-'?!.,#$%\'()*+-/:;<=>@[\\]^_`{|}~" + '""“”’' + '∞θ÷α•à−β∅³π‘₹´°£€\×™√²—–&'
    
    punct_mapping = {"‘": "'", "₹": "e", "´": "'", "°": "", "€": "e", "™": "tm", "√": " sqrt ", "×": "x", "²": "2", "—": "-", "–": "-", "’": "'", "_": "-",
                     "`": "'", '“': '"', '”': '"', '“': '"', "£": "e", '∞': 'infinity', 'θ': 'theta', '÷': '/', 'α': 'alpha', '•': '.', 'à': 'a', '−': '-', 
                     'β': 'beta', '∅': '', '³': '3', 'π': 'pi', }

    def clean_special_chars(text, punct, mapping):
        for p in mapping:
            text = text.replace(p, mapping[p])
        for p in punct:
            text = text.replace(p, f' {p} ')
        specials = {'\u200b': ' ', '…': ' ... ', '\ufeff': '', 'करना': '', 'है': ''}  
        for s in specials:
            text = text.replace(s, specials[s])
        return text

    df['clean_text'] = df['cleaned_text'].apply(lambda x: clean_special_chars(x, punct, punct_mapping))

    # def correct_spelling(text):
    #     """
    #     Corrects the spelling of words in the provided text.

    #     :param text: The text to be spell-checked.
    #     :return: The text with corrected spelling.
    #     """
    #     spell = SpellChecker()
    #     words = text.split()  # Tokenize the text into words
    #     corrected_words = [spell.correction(word) for word in words]
    #     corrected_text = ' '.join(corrected_words)
    #     return corrected_text
    
    # df['clean_text'] = df['clean_text'].apply(lambda x: correct_spelling(x))


    return df

In [11]:
# time the run time

import time

start = time.time()

# Clean the train and test text data

train = clean_text(train, col_name = 'full_text')


end = time.time()

print(f"Time taken to clean text: {end - start} seconds")

Starting text cleaning process. 

Time taken to clean text: 1.0322492122650146 seconds


In [12]:
glove_path = '/home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt'
paragran_path = '/home/laptop/github/kaggle/scoring/data/paragram-300-sl999.txt'
fastetxt_path = '/home/laptop/github/kaggle/scoring/data/wiki-news-1M-300d.vec'


# Rebuild and check vocab after cleaning contractions

train, glove, paragram, fastetxt = embedding_checks(train, glove_path, paragran_path, fastetxt_path, col_name='clean_text')

Loading all embeddings.
Loading embeddings from /home/laptop/github/kaggle/scoring/data/glove-840B-300d.txt


TypeError: 'module' object is not callable

In [ ]:
from transformers import pipeline, AutoTokenizer

# Initialize the BERT model and tokenizer
model_name = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
fill_mask = pipeline('fill-mask', model=model_name)

def bert_spell_check_chunked(text):
    max_length = tokenizer.model_max_length - 2  # Subtracting 2 for special tokens [CLS] and [SEP]
    tokens = tokenizer.tokenize(text)
    chunk_size = max_length

    # Splitting tokens into chunks
    chunks = [tokens[i:i + chunk_size] for i in range(0, len(tokens), chunk_size)]

    corrected_text = ''
    for chunk in chunks:
        chunk_text = tokenizer.convert_tokens_to_string(chunk)
        masked_sentence = chunk_text.replace(chunk_text, '[MASK]')
        prediction = fill_mask(masked_sentence)[0]['sequence']
        corrected_text += prediction + ' '

    return corrected_text

# Applying the function to each row of your DataFrame column
df['corrected_text'] = df['clean_text'].apply(bert_spell_check_chunked)


In [70]:
print(glove_train, paragram_train, fastetxt_train)
print(len(glove_train + paragram_train + fastetxt_train))

[('driveless', 1603), ('vauban', 1189), ('unrra', 476), ('bomberger', 352), ('eckman', 351), ('selsky', 299), ('heidrun', 297), ('¨', 139), ('elctoral', 129), ('diverless', 107), ('\x94', 106), ('valledupar', 92), ('drivless', 88), ('risorius', 73), ('landfrom', 66), ('venuse', 59), ('lanform', 53), ('haung', 51), ('sivak', 49), ('mockus', 46), ('eletoral', 46), ('palpabraeus', 44), ('landformation', 43), ('paragrapgh', 43), ('venuses', 41), ('vaubans', 39), ('thrun', 36), ('electorals', 35), ('elctors', 34), ('ellectoral', 32), ('heidrum', 31), ('elecoral', 31), ('technogly', 30), ('antanas', 30), ('paragrpah', 29), ('pursit', 29), ('stanislavsky', 29), ('inconclusion', 28), ('limting', 27), ('tempetures', 26), ('vensus', 25), ('electoal', 25), ('pursuiting', 24), ('electroal', 24), ('electral', 24), ('dirverless', 23), ('facail', 22), ('scienctist', 22), ('atomospheric', 22), ('driverles', 22), ('selfdriving', 21), ('paragrph', 21), ('ailens', 21), ('coyboys', 20), ('palnet', 20), ('

In [71]:
train.head()

,essay_id,full_text,score,lowered_question,cleaned_text,clean_text,lowered
1,000fe60,I am a scientist at NASA that is discussing th...,3,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...,i am a scientist at nasa that is discussing th...
2,001ab80,People always wish they had the same technolog...,4,people always wish they had the same technolog...,people always wish they had the same technolog...,people always wish they had the same technolog...,people always wish they had the same technolog...
3,001bdc0,"We all heard about Venus, the planet without a...",4,"we all heard about venus, the planet without a...","we all heard about venus, the planet without a...","we all heard about venus , the planet without...","we all heard about venus, the planet without a..."
4,002ba53,"Dear, State Senator\n\nThis is a letter to arg...",3,"dear, state senator\n\nthis is a letter to arg...","dear, state senator\n\nthis is a letter to arg...","dear , state senator\n\nthis is a letter to a...","dear, state senator\n\nthis is a letter to arg..."
6,0033037,The posibilty of a face reconizing computer wo...,2,the posibilty of a face reconizing computer wo...,the posibilty of a face reconizing computer wo...,the posibilty of a face reconizing computer wo...,the posibilty of a face reconizing computer wo...


In [17]:
print(train['full_text'][0])

Many people have car where they live. The thing they don't know is that when you use a car alot of thing can happen like you can get in accidet or the smoke that the car has is bad to breath on if someone is walk but in VAUBAN,Germany they dont have that proble because 70 percent of vauban's families do not own cars,and 57 percent sold a car to move there. Street parkig ,driveways and home garages are forbidden on the outskirts of freiburd that near the French and Swiss borders. You probaly won't see a car in Vauban's streets because they are completely "car free" but If some that lives in VAUBAN that owns a car ownership is allowed,but there are only two places that you can park a large garages at the edge of the development,where a car owner buys a space but it not cheap to buy one they sell the space for you car for $40,000 along with a home. The vauban people completed this in 2006 ,they said that this an example of a growing trend in Europe,The untile states and some where else ar

In [18]:
print(train['clean_text'][0])

many people have car where they live .  the thing they do not know is that when you use a car alot of thing can happen like you can get in accident or the smoke that the car has is bad to breath on if someone is walk but in  , germany they dont have that proble because 70 percent of   '  s families do not own cars , and 57 percent sold a car to move there .  street parking  , driveways and home garages are forbidden on the outskirts of  that near the french and swiss borders .  you probaly will not see a car in   '  s streets because they are completely   "  car free  "   but if some that lives in  that owns a car ownership is allowed , but there are only two places that you can park a large garages at the edge of the development , where a car owner buys a space but it not cheap to buy one they sell the space for you car for  $ 40 , 000 along with a home .  the  people completed this in 2006  , they said that this an example of a growing trend in europe , the untile states and some whe

In [ ]:
def custom_train_validation_split(essays, test_size=0.2, random_state=56):
    
    """
    Custom function to perform train-validation split ensuring that
    the same prompt IDs are in both training and validation sets.

    Parameters:
    - summaries: DataFrame containing summaries and associated prompt_ids
    - prompts: DataFrame containing prompts and associated prompt_ids
    - test_size: Proportion of the dataset to be used as the validation set
    - random_state: Random seed for reproducibility

    Returns:
    - train_summaries: Training set containing summaries
    - validation_summaries: Validation set containing summaries
    - train_prompts: Training set containing prompts
    - validation_prompts: Validation set containing prompts
    """
    from sklearn.model_selection import train_test_split
    # Extract unique prompt IDs
    unique_essay_ids = essays['essay_id'].unique()

    # Split the unique prompt IDs into training and validation sets
    train_ids, validation_ids = train_test_split(unique_essay_ids, test_size=test_size, random_state=random_state)

    # Use these IDs to filter the original summaries and prompts DataFrames
    train_essays = essays[essays['essay_id'].isin(train_ids)]
    validation_essays = essays[essays['essay_id'].isin(validation_ids)]

    return train_essays, validation_essays


train, validation = custom_train_validation_split(train, 0.25)

In [19]:
# Save the cleaned text data to a Parquet file

train.to_parquet('clean_train.parquet')
validation.to_parquet('clean_validation.parquet')

# test.to_parquet('cleaned_test.parquet')

In [ ]:
# save